## Zillow Data Preprocessing

This notebook prepares county-level Zillow Home Value Index (ZHVI) data
for analysis by aggregating monthly home value estimates into a single
annual measure for 2022.

The resulting dataset will serve as the dependent variable in the housing
price analysis.


### Step 1. Import Required Libraries

We import the core libraries required for file handling and data manipulation.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


### Step 2. Load the Raw Zillow Data

The raw Zillow dataset contains monthly home value estimates for U.S.
counties over a long time span. We load the dataset and perform basic
checks to understand its structure.


In [2]:
ZILLOW_RAW = RAW_DIR / "zillow" / "County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
df = pd.read_csv(ZILLOW_RAW)
df.shape


(3073, 320)

In [3]:
df.head()


,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,2000-01-31,...,2025-02-28,2025-03-31,2025-04-30,2025-05-31,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30
0,3101,0,Los Angeles County,county,CA,CA,"Los Angeles-Long Beach-Anaheim, CA",6,37,206254.019343,...,874964.917945,868183.050256,862929.567146,857985.595709,853866.432963,851978.709943,851943.653835,854006.515744,856787.549041,859957.764091
1,139,1,Cook County,county,IL,IL,"Chicago-Naperville-Elgin, IL-IN-WI",17,31,146779.748575,...,308030.174553,308520.192845,308782.399434,308769.740342,308780.748645,309505.428049,310406.060493,311748.721178,313031.329872,314516.514574
2,1090,2,Harris County,county,TX,TX,"Houston-The Woodlands-Sugar Land, TX",48,201,110782.116581,...,284440.985204,283636.284757,282607.106673,281611.089122,280572.592024,279717.466972,278968.206579,278419.642992,277968.782059,277739.656060
3,2402,3,Maricopa County,county,AZ,AZ,"Phoenix-Mesa-Chandler, AZ",4,13,144805.489877,...,468538.235977,466370.415404,464363.291204,462328.972349,460446.819030,458424.482456,456768.337428,455696.185830,455267.766429,455489.787586
4,2841,4,San Diego County,county,CA,CA,"San Diego-Chula Vista-Carlsbad, CA",6,73,212987.895508,...,934604.371701,932496.828270,929363.457823,924579.270022,919383.349850,915040.665350,911645.941022,909804.606498,909773.314298,910764.623616


### Step 3. Verify Geographic Level

This project focuses on county-level analysis. As a sanity check,
we confirm that all observations in the dataset correspond to counties.


In [4]:
df["RegionType"].value_counts()


RegionType
county    3073
Name: count, dtype: int64

### Step 4. Identify Monthly Columns for 2022

In this Zillow file, monthly observations are stored as columns named in ISO format
(`YYYY-MM-DD`), such as `2022-01-31`. Therefore, we select all columns that start
with `2022-`, which should correspond to the 12 monthly values in 2022.


In [5]:
date_cols_2022 = [c for c in df.columns if c.startswith("2022-")]
sorted(date_cols_2022)


['2022-01-31',
 '2022-02-28',
 '2022-03-31',
 '2022-04-30',
 '2022-05-31',
 '2022-06-30',
 '2022-07-31',
 '2022-08-31',
 '2022-09-30',
 '2022-10-31',
 '2022-11-30',
 '2022-12-31']

### Step 5. Compute the 2022 Annual Average ZHVI

To align Zillow's monthly home value series with annual ACS socioeconomic variables,
we aggregate the 12 monthly ZHVI values in 2022 into a single annual average for each county.
This reduces short-term volatility and improves conceptual comparability across data sources.



In [6]:
df["zhvi_2022"] = df[date_cols_2022].mean(axis=1)
df[["RegionName", "State", "zhvi_2022"]].head()


,RegionName,State,zhvi_2022
0,Los Angeles County,CA,822473.001114
1,Cook County,IL,282612.668412
2,Harris County,TX,279352.397555
3,Maricopa County,AZ,483807.870034
4,San Diego County,CA,850809.443854


### Step 6. Create County FIPS for Future Merging

To merge Zillow home values with ACS/Census data reliably, we create a standardized
5-digit County FIPS code. This avoids common issues caused by inconsistent county name
formats across sources.


In [7]:
df["state_fips"] = df["StateCodeFIPS"].astype(str).str.zfill(2)
df["county_fips"] = df["MunicipalCodeFIPS"].astype(str).str.zfill(3)
df["fips"] = df["state_fips"] + df["county_fips"]

df[["State", "RegionName", "state_fips", "county_fips", "fips"]].head()


,State,RegionName,state_fips,county_fips,fips
0,CA,Los Angeles County,06,037,06037
1,IL,Cook County,17,031,17031
2,TX,Harris County,48,201,48201
3,AZ,Maricopa County,04,013,04013
4,CA,San Diego County,06,073,06073


### Step 7. Select Analysis-Ready Columns and Handle Missing Values

We keep only the geographic identifiers and the 2022 annual home value measure
required for downstream analysis.

Puerto Rico counties (state FIPS 72) are excluded to maintain consistency with ACS county coverage.

Finally, counties with missing 2022 ZHVI values are removed because the target
variable cannot be defined for those observations.


In [8]:
zillow_2022 = df[["fips", "State", "RegionName", "zhvi_2022"]].copy()

zillow_2022["fips"] = zillow_2022["fips"].astype(str).str.zfill(5)
zillow_2022 = zillow_2022[~zillow_2022["fips"].str.startswith("72")].copy()

zillow_2022.isna().sum()


fips           0
State          0
RegionName     0
zhvi_2022     12
dtype: int64

In [9]:
zillow_2022 = zillow_2022.dropna(subset=["zhvi_2022"]).reset_index(drop=True)
zillow_2022.shape, zillow_2022.head()


((3061, 4),
     fips State          RegionName      zhvi_2022
 0  06037    CA  Los Angeles County  822473.001114
 1  17031    IL         Cook County  282612.668412
 2  48201    TX       Harris County  279352.397555
 3  04013    AZ     Maricopa County  483807.870034
 4  06073    CA    San Diego County  850809.443854)

### Step 8. Save the Processed Dataset

We save the county-level dataset (with County FIPS and 2022 annual average ZHVI)
to the `data/processed` directory for use in EDA and modeling.


In [10]:
ZILLOW_OUTPUT = "zillow_zhvi_2022.csv"

zillow_2022.to_csv(PROCESSED_DIR / ZILLOW_OUTPUT, index=False)
print(f"Saved: data/processed/{ZILLOW_OUTPUT}")


Saved: data/processed/zillow_zhvi_2022.csv


### Summary

This notebook transforms Zillow monthly ZHVI county data into a single
county-level target variable by computing the 2022 annual average home value
(`zhvi_2022`). A standardized 5-digit County FIPS code is also constructed
to support reliable merging with ACS/Census datasets.

A small number of U.S. counties (12) did not have available ZHVI values for 2022.
These observations were excluded from the analysis because the target variable
could not be defined for those regions.
